In [26]:
###### Config #####
import sys, os, platform
if os.path.isdir("ds-assets"):
  !cd ds-assets && git pull
else:
  !git clone https://github.com/lutzhamel/ds-assets.git
colab = True if 'google.colab' in os.sys.modules else False
system = platform.system() # "Windows", "Linux", "Darwin"
home = "ds-assets/assets/"
sys.path.append(home)  

2861.82s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Already up to date.


In [27]:
# notebook level imports
import pandas as pd
import numpy as np
import dsutils
from sklearn import tree
from sklearn import model_selection
from sklearn import metrics

# Review

In the material we have covered we have introduced three main ideas:

1. We need to **search for optimal models** using separate training and testing sets.  Using only a training set will produce **overfitted models** or in other words models that are overly optimistic.  The search means using the model parameters (sometimes called hyperparameters) to control the model complexity.

1. Once we have a trained, optimal model we need to **evaluate its performance**.  For **classification** models this means computing the **accuracy** and for **regression** models this means computing the **$R^2$** score.  For classification problems we also want to compute the **confusion matrix**.  For binary classification problems this allows us to look at the false positive/false negative rates (type I/type II errors) of the model which is especially critical in clinical settings. For non-binary classification problems the confusion matrix allows us to see which labels were the most difficult to classify for the model.

1. The third idea centers around **confidence intervals** and **statistical significance**.  Confidence intervals give us a measure of the underlying **data uncertainty** and statistical significance allows us to evaluate if the performances of two models are truly different.  This holds for both classification and regression models.

# Constructing Optimal Models

Constructing optimal models can be summarized by the **learning curves plot**.

<img src="https://raw.githubusercontent.com/lutzhamel/ds-assets/main/assets/train-test-curves.png"  height="300" width="450">

This means we search for models by modifying its hyperparameters to vary complexity and train and test with the appropriate data partitions.

The most convenient way to do this using the **gridsearch** together with the **refit error**.



Let's try this with our Iris dataset.

In [28]:
df = pd.read_csv(home+"iris.csv")
X  = df.drop(columns=['id','Species'])
y = df[['Species']]

In [29]:
## setting up grid search

# find the maximum depth for our tree model
depth_ceiling = tree.DecisionTreeClassifier().fit(X, y).get_depth()

# create an untrained tree model object for the gridsearch
model = tree.DecisionTreeClassifier()

# set up our parameter search space
param = {
   'max_depth': list(range(1,depth_ceiling+1)),
   'criterion': ['gini', 'entropy']
   }  

# find the best model            
best_model = model_selection.GridSearchCV(model, param).fit(X,y)

In [30]:
print(best_model.best_params_)

{'criterion': 'gini', 'max_depth': 4}


# Evaluating Models

Here we want to look at the accuracy and where the model makes its mistakes.

In [31]:
print(best_model.score(X,y))

0.9933333333333333


In [32]:
dsutils.confusion_matrix(best_model, X, y, labels=list(y['Species'].unique()))

,setosa,versicolor,virginica
setosa,50,0,0
versicolor,0,50,0
virginica,0,1,49


# Confidence Intervals

Confidence intervals tell us something about the underlying uncertainties of the data.  When reporting results you should always include the confidence interval.

In [33]:
print(dsutils.acc_score(best_model,X,y,as_string=True))

Accuracy: 0.99 (0.98, 1.00)


The interesting thing about confidence intervals is that they allow us to determine if the performance difference between two classifiers is statistically significant or not.

**Exercise**: Build a decision tree classifier limited to a depth of one and determine if its performance is statistically significantly different from the optimal tree we constructed above.